# Ore classifier 3-class training

Maps `thin` and `refractory` to one `difficult` class. Logs training to wandb and exports the best model.

In [6]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import wandb
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

ROOT = Path('/home/team117/nornik')
MANIFEST = ROOT / 'ore_3class' / 'manifest.csv'
MODEL_NAME = 'efficientnet_b3'
IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 25
SEED = 42
CLASSES = ['ordinary', 'difficult', 'talc']
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
idx_to_label = {i: c for c, i in label_to_idx.items()}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [7]:
df = pd.read_csv(MANIFEST)
print(df['label_3class'].value_counts())
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label_3class'],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(len(train_df), len(val_df))

label_3class
ordinary     540
difficult    458
talc         126
Name: count, dtype: int64
899 225


In [8]:
class OreDataset(Dataset):
    def __init__(self, frame, tfm):
        self.frame = frame
        self.tfm = tfm
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(ROOT / row['path']).convert('RGB')
        return self.tfm(image), label_to_idx[row['label_3class']]

train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
train_loader = DataLoader(OreDataset(train_df, train_tfm), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(OreDataset(val_df, val_tfm), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [9]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(device)
counts = train_df['label_3class'].value_counts().reindex(CLASSES).values.astype(np.float32)
weights = torch.tensor(counts.sum() / np.maximum(counts, 1), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
run = wandb.init(project='nornik-ore-classifier', name=f'{MODEL_NAME}-3class-512', config={
    'model': MODEL_NAME, 'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'classes': CLASSES, 'mapping': {'thin': 'difficult', 'refractory': 'difficult'},
})

epoch,▁
train/accuracy,▁
train/balanced_accuracy,▁
train/loss,▁
train/macro_f1,▁
val/accuracy,▁
val/balanced_accuracy,▁
val/loss,▁
val/macro_f1,▁
epoch,1
train/accuracy,0.68966


In [10]:
def run_epoch(loader, train: bool, epoch: int):
    model.train(train)
    losses, y_true, y_pred = [], [], []
    phase = 'train' if train else 'val'
    progress = tqdm(loader, desc=f'{phase} {epoch}', leave=False)
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        losses.append(loss.item())
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
        progress.set_postfix(loss=f'{np.mean(losses):.4f}')
    return {
        'loss': float(np.mean(losses)),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'cm': confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES)))),
        'y_true': y_true,
        'y_pred': y_pred,
    }

best_f1 = -1
out_dir = ROOT / 'models'
out_dir.mkdir(exist_ok=True)
best_path = out_dir / 'ore_classifier_3class_effb3.pt'
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, True, epoch)
    val_metrics = run_epoch(val_loader, False, epoch)
    scheduler.step()
    log = {f'train/{k}': v for k, v in train_metrics.items() if k not in {'cm', 'y_true', 'y_pred'}}
    log.update({f'val/{k}': v for k, v in val_metrics.items() if k not in {'cm', 'y_true', 'y_pred'}})
    log['epoch'] = epoch
    wandb.log(log)
    print(epoch, log)
    if val_metrics['macro_f1'] > best_f1:
        best_f1 = val_metrics['macro_f1']
        torch.save({'model': model.state_dict(), 'classes': CLASSES, 'model_name': MODEL_NAME, 'img_size': IMG_SIZE}, best_path)
        wandb.log({
            'val/confusion_matrix': wandb.plot.confusion_matrix(
                probs=None,
                y_true=val_metrics['y_true'],
                preds=val_metrics['y_pred'],
                class_names=CLASSES,
            )
        })
print('best', best_f1, best_path)

1 {'train/loss': 0.9594571638390818, 'train/accuracy': 0.71412680756396, 'train/balanced_accuracy': 0.7213407816191456, 'train/macro_f1': 0.6878446039693551, 'val/loss': 0.9993887559093279, 'val/accuracy': 0.6622222222222223, 'val/balanced_accuracy': 0.7396188942565755, 'val/macro_f1': 0.6360664157236157, 'epoch': 1}


2 {'train/loss': 0.5769525382222723, 'train/accuracy': 0.7830923248053393, 'train/balanced_accuracy': 0.8004748106868974, 'train/macro_f1': 0.7650344439022829, 'val/loss': 0.47981155830724487, 'val/accuracy': 0.8311111111111111, 'val/balanced_accuracy': 0.842313472893183, 'val/macro_f1': 0.8051104637577668, 'epoch': 2}


3 {'train/loss': 0.40958403415186745, 'train/accuracy': 0.8576195773081201, 'train/balanced_accuracy': 0.8681553720035028, 'train/macro_f1': 0.8472016048731984, 'val/loss': 0.4991065884201691, 'val/accuracy': 0.8266666666666667, 'val/balanced_accuracy': 0.8526945786366076, 'val/macro_f1': 0.8004652437960571, 'epoch': 3}


4 {'train/loss': 0.35169848389470276, 'train/accuracy': 0.8832035595105673, 'train/balanced_accuracy': 0.8872937043248951, 'train/macro_f1': 0.8699550369996626, 'val/loss': 0.32972379374028793, 'val/accuracy': 0.8533333333333334, 'val/balanced_accuracy': 0.8696027911969941, 'val/macro_f1': 0.8353992127408719, 'epoch': 4}


5 {'train/loss': 0.27691053661653786, 'train/accuracy': 0.8921023359288098, 'train/balanced_accuracy': 0.8994979871393333, 'train/macro_f1': 0.8856164012572423, 'val/loss': 0.33980635417412547, 'val/accuracy': 0.8888888888888888, 'val/balanced_accuracy': 0.9061513687600643, 'val/macro_f1': 0.8708002725192862, 'epoch': 5}


...

21 {'train/loss': 0.014971189595325191, 'train/accuracy': 0.9955506117908788, 'train/balanced_accuracy': 0.9969135802469135, 'train/macro_f1': 0.9966380182002021, 'val/loss': 0.29069787486261295, 'val/accuracy': 0.9244444444444444, 'val/balanced_accuracy': 0.9141062801932366, 'val/macro_f1': 0.9050079183895744, 'epoch': 21}


22 {'train/loss': 0.03238262012076772, 'train/accuracy': 0.9866518353726362, 'train/balanced_accuracy': 0.9851267230548192, 'train/macro_f1': 0.9874292598460782, 'val/loss': 0.2758495173304082, 'val/accuracy': 0.92, 'val/balanced_accuracy': 0.9212667740203972, 'val/macro_f1': 0.903358691479259, 'epoch': 22}


23 {'train/loss': 0.020014669879703306, 'train/accuracy': 0.9933259176863182, 'train/balanced_accuracy': 0.9927025034015241, 'train/macro_f1': 0.9924963626278008, 'val/loss': 0.2969210428253001, 'val/accuracy': 0.92, 'val/balanced_accuracy': 0.9218035426731079, 'val/macro_f1': 0.9034021587589974, 'epoch': 23}


24 {'train/loss': 0.056219647714865095, 'train/accuracy': 0.982202447163515, 'train/balanced_accuracy': 0.9789550107105428, 'train/macro_f1': 0.9827585850054001, 'val/loss': 0.3144240238670418, 'val/accuracy': 0.9066666666666666, 'val/balanced_accuracy': 0.9125442834138487, 'val/macro_f1': 0.8830918638846524, 'epoch': 24}


25 {'train/loss': 0.034855521198084466, 'train/accuracy': 0.9833147942157954, 'train/balanced_accuracy': 0.9822553407435461, 'train/macro_f1': 0.9812972362190832, 'val/loss': 0.3539865838553443, 'val/accuracy': 0.8933333333333333, 'val/balanced_accuracy': 0.8935748792270531, 'val/macro_f1': 0.8676561533704391, 'epoch': 25}
best 0.905684581874549 /home/team117/nornik/models/ore_classifier_3class_effb3.pt


In [11]:
# Export to ONNX after training.
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_path = ROOT / 'models' / 'ore_classifier_3class_effb3.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
metadata = {'classes': CLASSES, 'model_name': MODEL_NAME, 'img_size': IMG_SIZE, 'checkpoint': str(best_path), 'onnx': str(onnx_path)}
(ROOT / 'models' / 'ore_classifier_3class_effb3.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(onnx_path)

/tmp/ipykernel_33398/1257537633.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_path, map_location=device)


/home/team117/nornik/models/ore_classifier_3class_effb3.onnx


In [12]:
val_df

,path,source_folder,label_3class,label_ru
0,remaining_dataset_clean/images_by_class/thin/t...,thin,difficult,Труднообогатимая руда
1,remaining_dataset_clean/images_by_class/talc/t...,talc,talc,Оталькованная руда
2,remaining_dataset_clean/images_by_class/thin/t...,thin,difficult,Труднообогатимая руда
3,remaining_dataset_clean/images_by_class/talc/t...,talc,talc,Оталькованная руда
4,remaining_dataset_clean/images_by_class/thin/t...,thin,difficult,Труднообогатимая руда
...,...,...,...,...
220,remaining_dataset_clean/images_by_class/ordina...,ordinary,ordinary,Рядовая руда
221,remaining_dataset_clean/images_by_class/thin/t...,thin,difficult,Труднообогатимая руда
222,remaining_dataset_clean/images_by_class/thin/t...,thin,difficult,Труднообогатимая руда
223,remaining_dataset_clean/images_by_class/talc/t...,talc,talc,Оталькованная руда
